# Module 05: Distributed Systems — Hadoop to EMR

**Program:** Quintrix Jr. Data Engineer Training
**Duration:** 3 hours
**Instructor:** Sunil Mogadati

---

**Today's Agenda:**

| # | Topic | Time |
|---|-------|------|
| 1 | Why Distributed Systems? (The Pizza Problem) | 20 min |
| 2 | HDFS — The Distributed File System | 20 min |
| 3 | MapReduce — The Original Processing Model | 25 min |
| 4 | YARN — The Resource Manager | 15 min |
| 5 | Hive — SQL on Hadoop | 25 min |
| 6 | The Legacy-to-Modern Mapping | 15 min |
| 7 | AWS EMR — Managed Hadoop in the Cloud | 25 min |
| 8 | GCP Dataproc — Google's Equivalent | 10 min |
| 9 | Hands-On: The EMR Workflow | 20 min |
| 10 | Key Takeaways + Homework | 5 min |

---

> **Why this module matters:** Every managed cloud data service you'll use — EMR, Dataproc, Athena, BigQuery — is built on the architecture invented by the Hadoop ecosystem. You're not learning dead tech. You're learning the blueprint that everything modern runs on.

In [ ]:
# ============================================================# HELLO WORLD — MapReduce in 10 Lines of Python# ============================================================# This is MapReduce: split the work (map), combine the results (reduce).# Hadoop does this across 1000 machines. We'll do it on one.from collections import Counter  # Counter = a dict that counts things# Our "document" — imagine this is 100 TB of text across 1000 machinesdocument = "the quick brown fox jumps over the lazy dog the fox the dog"# MAP step: split the document into individual words# WHY: Each machine gets a chunk of text and splits its local piecewords = document.split()  # .split() breaks a string on whitespace# REDUCE step: count occurrences of each word# WHY: After mapping, we combine (reduce) partial counts into totalsword_counts = Counter(words)  # Counter does the reduce automatically# Output — sorted by frequency (most common first)print("=== MapReduce Word Count ===")for word, count in word_counts.most_common():    print(f"  {word}: {count}")# You should see:#   the: 4#   fox: 2#   dog: 2#   quick: 1  ... etc.## This tiny script IS MapReduce. Hadoop just does it across 1000 machines# with fault tolerance, replication, and network shuffling.

---

## 1. Why Distributed Systems? (The Pizza Problem)

**One-line answer:** When a single machine can't handle the data, you split the work across many machines.

### The Pizza Analogy

Imagine you run a pizza shop. On a normal night, one chef handles all the orders. But on Super Bowl Sunday, 500 orders come in at once. You have two choices:

| Strategy | What It Means | In Computing |
|----------|--------------|--------------|
| **Get a bigger oven** | Buy a $50,000 industrial oven that fits 50 pizzas | **Vertical scaling** — buy a bigger server (more CPU, more RAM) |
| **Hire more chefs** | Hire 10 chefs, each with their own oven, split the orders | **Horizontal scaling** — add more machines, split the data |

Vertical scaling has a ceiling — there's a limit to how big one oven (or one server) can get. Horizontal scaling has no ceiling — you can always add more chefs.

> **Hadoop's big idea:** Instead of one $1 million supercomputer, use 1,000 $1,000 **commodity machines** (regular, off-the-shelf servers — nothing special). Cheaper, more resilient, and scales infinitely.

### The Scale Problem (Real Numbers)

| Scenario | Data Size | Single Machine | Distributed (100 nodes) |
|----------|-----------|---------------|------------------------|
| Read a CSV | 100 MB | 2 seconds | Overkill |
| Daily call center logs | 10 GB | 5 minutes | 3 seconds |
| A year of call data | 3 TB | Won't fit in RAM | 30 seconds |
| Netflix viewing history | 100 TB | Impossible | 5 minutes |
| Google web index | 100+ PB | Laughable | This is why Google built it |

### The Three Problems Hadoop Solves

```
Problem 1: STORAGE — Where do you put 100 TB of data?
  Answer:  HDFS — split it across 100 machines

Problem 2: PROCESSING — How do you compute over 100 TB?
  Answer:  MapReduce/Spark — each machine processes its local chunk

Problem 3: COORDINATION — Who decides which machine does what?
  Answer:  YARN — the resource manager that assigns work
```

> **Think of it as a warehouse:** HDFS is the shelving system (where things are stored). MapReduce/Spark is the workforce (who does the work). YARN is the warehouse manager (who assigns tasks to workers).

In [ ]:
# ============================================================
# WHY DISTRIBUTED? — The Scale Intuition
# ============================================================

# How long would it take to process data on 1 machine vs many?
# WHY: This shows why distributed systems exist — single machines can't keep up
import math

# These sizes represent real-world scenarios — from laptop to enterprise
data_sizes = {
    "100 MB (small CSV)": 100,
    "10 GB (daily logs)": 10_000,
    "1 TB (monthly data)": 1_000_000,
    "100 TB (data lake)": 100_000_000,
    "1 PB (enterprise)": 1_000_000_000,
}

# Assume: 1 machine processes 200 MB/sec (realistic for disk I/O)
single_machine_speed = 200  # MB/sec — realistic for sequential disk I/O (read speed of one HDD)  # MB/sec

print(f"{'Data Size':<25} {'1 Machine':>15} {'10 Nodes':>15} {'100 Nodes':>15}")
print("-" * 72)
for label, size_mb in data_sizes.items():
    t1 = size_mb / single_machine_speed         # t1 = time on 1 machine
    t10 = size_mb / (single_machine_speed * 10)  # t10 = time on 10 machines (linear speedup)
    t100 = size_mb / (single_machine_speed * 100) # t100 = time on 100 machines

    # fmt = format seconds into human-readable time strings
    def fmt(seconds):
        if seconds < 60:
            return f"{seconds:.1f}s"
        elif seconds < 3600:
            return f"{seconds/60:.1f} min"
        elif seconds < 86400:
            return f"{seconds/3600:.1f} hrs"
        else:
            return f"{seconds/86400:.1f} days"

    print(f"{label:<25} {fmt(t1):>15} {fmt(t10):>15} {fmt(t100):>15}")

print("\nThis is the power of horizontal scaling.")
print("Hadoop made it possible. Cloud services (EMR, Dataproc) made it easy.")

---

## 2. HDFS — The Distributed File System

**One-line answer:** HDFS splits every file into 128 MB blocks and stores each block on 3 different machines, so nothing is ever lost.

### The Library Analogy

Imagine a library that has a single copy of every book. If that shelf catches fire, the book is gone forever. Now imagine a library system with 3 branches — every book exists in all 3 locations. One branch floods? The book still exists in the other two.

That's HDFS. Every block of data is replicated 3 times across different machines (the **replication factor**).

### How HDFS Works

```
You write a 512 MB file to HDFS:

Step 1: Split into blocks (default 128 MB each)
  Block 1: 128 MB
  Block 2: 128 MB
  Block 3: 128 MB
  Block 4: 128 MB

Step 2: Distribute + Replicate (replication factor = 3)
  +----------+----------+----------+----------+
  | Machine A| Machine B| Machine C| Machine D|
  +----------+----------+----------+----------+
  | Block 1  | Block 1  | Block 2  | Block 3  |
  | Block 3  | Block 2  | Block 4  | Block 1  |
  | Block 4  | Block 3  |          | Block 2  |
  +----------+----------+----------+----------+

Machine B dies? No data lost — every block exists on 2 other machines.
HDFS automatically re-replicates the lost copies to maintain factor of 3.
```

### NameNode vs DataNode

| Component | Role | Analogy |
|-----------|------|---------|
| **NameNode** | Keeps the directory — knows which blocks are on which machines | The librarian's card catalog |
| **DataNode** | Stores the actual data blocks | The bookshelves in each branch |

```
NameNode (1 per cluster):
  "File: /data/calls.parquet"
  "Block 1 → Machine A, Machine B, Machine D"
  "Block 2 → Machine B, Machine C, Machine D"
  ...

DataNodes (many per cluster):
  Each stores actual 128 MB blocks on local disk
```

> **Critical point:** The NameNode is a single point of failure. If it dies, the cluster is blind — it doesn't know where anything is. That's why production clusters always have a **Standby NameNode** (hot backup).

### HDFS vs Modern Cloud Storage

| Feature | HDFS | S3 / GCS |
|---------|------|----------|
| **Where** | On-prem cluster | Cloud (managed) |
| **Coupled to compute?** | Yes — data sits on the same machines that process it | No — storage is separate from compute |
| **Replication** | 3x by default (you manage it) | "11 nines" durability — 99.999999999% (they manage it) |
| **Cost** | You buy the servers | Pay per GB stored |
| **Scaling** | Add more DataNodes (hardware) | Infinite (just upload more) |
| **Maintenance** | You fix failed disks, rebalance data | Zero — fully managed |

> **The big shift:** Hadoop required data and compute on the same machines. Cloud separated them — store on S3, process with EMR. This is why cloud won.

In [ ]:
# ============================================================
# HDFS — Block Distribution Simulation
# ============================================================
# WHY: This simulation shows how HDFS distributes and replicates data blocks
import random
random.seed(42)  # Fixed seed = reproducible results (same output every run)

def simulate_hdfs(file_size_mb, block_size_mb=128, replication=3, num_machines=5):
    """Simulate how HDFS distributes blocks across machines."""
    num_blocks = -(-file_size_mb // block_size_mb)  # ceiling division trick: 512/128 = 4 blocks  # ceiling division
        # DataNode = a machine that stores HDFS data blocks
    machines = {f"DataNode-{i+1}": [] for i in range(num_machines)}
    machine_names = list(machines.keys())

    print(f"File size: {file_size_mb} MB")
    print(f"Block size: {block_size_mb} MB")
    print(f"Blocks: {num_blocks}")
    print(f"Replication factor: {replication}")
    print(f"Machines: {num_machines}")
    print()

    for block_id in range(1, num_blocks + 1):
        # Pick 'replication' random machines for this block
                # Pick random machines for this block (simulates HDFS rack-aware placement)
        targets = random.sample(machine_names, min(replication, num_machines))
        for target in targets:
            machines[target].append(f"B{block_id}")

    # Display distribution
    print("Block Distribution:")
    print("-" * 50)
    for machine, blocks in machines.items():
        block_str = ", ".join(blocks) if blocks else "(empty)"
        storage_mb = len(blocks) * block_size_mb
        print(f"  {machine}: [{block_str}] = {storage_mb} MB")

    total_stored = sum(len(b) for b in machines.values()) * block_size_mb
    print(f"\nTotal stored: {total_stored} MB (= {file_size_mb} MB x {replication} replicas)")

    # Simulate machine failure
    # WHY: This proves replication works — lose a machine, keep all data
    failed = random.choice(machine_names)
    lost_blocks = set(machines[failed])
    surviving_blocks = set()  # Blocks that still exist on other machines
    for name, blocks in machines.items():
        if name != failed:
            surviving_blocks.update(blocks)
    truly_lost = lost_blocks - surviving_blocks

    print(f"\nSimulate failure: {failed} goes down")
    print(f"  Blocks on failed machine: {sorted(lost_blocks)}")
    print(f"  All blocks still available elsewhere: {len(truly_lost) == 0}")
    if truly_lost:
        print(f"  DATA LOST: {truly_lost}")
    else:
        print(f"  No data lost! Replication saved us.")

simulate_hdfs(file_size_mb=512, block_size_mb=128, replication=3, num_machines=5)

---

## 3. MapReduce — The Original Processing Model

**One-line answer:** MapReduce processes data in two steps — Map (transform each piece in parallel) and Reduce (combine the results).

### The Vote Counting Analogy

An election just happened. You have 10 million paper ballots to count. How do you do it?

**The naive way:** One person reads all 10 million ballots. Takes weeks.

**The MapReduce way:**

```
STEP 1: MAP (parallel) — hand ballots to 100 volunteers
  Volunteer 1: reads their stack → "Candidate A: 47, Candidate B: 53"
  Volunteer 2: reads their stack → "Candidate A: 51, Candidate B: 49"
  Volunteer 3: reads their stack → "Candidate A: 44, Candidate B: 56"
  ...

STEP 2: SHUFFLE — group all counts by candidate
  Candidate A: [47, 51, 44, ...]
  Candidate B: [53, 49, 56, ...]

STEP 3: REDUCE — add up each candidate's total
  Candidate A: 47 + 51 + 44 + ... = 4,812,000
  Candidate B: 53 + 49 + 56 + ... = 5,188,000

Result: Candidate B wins.
```

Each volunteer works **independently** on their local stack — that's the power. No one needs to see anyone else's ballots.

### MapReduce in Data Engineering

```
Example: "What is the total revenue per campaign?"

Input: 100 TB of call records across 100 machines

MAP phase (runs on each machine locally):
  Machine 1: reads its local blocks
    → ("Solar", 149.99), ("Auto", 0), ("Solar", 89.50), ...
  Machine 2: reads its local blocks
    → ("Medicare", 210.00), ("Solar", 125.00), ...
  ...

SHUFFLE phase (network transfer — the expensive part):
  All "Solar" values  → sent to Reducer 1
  All "Auto" values   → sent to Reducer 2
  All "Medicare" values → sent to Reducer 3

REDUCE phase:
  Reducer 1: sum("Solar")    = $2,450,000
  Reducer 2: sum("Auto")     = $890,000
  Reducer 3: sum("Medicare") = $3,100,000
```

### Why MapReduce Lost to Spark

| Aspect | MapReduce | Spark |
|--------|-----------|-------|
| **Between steps** | Writes to disk (HDFS) | Keeps data in memory |
| **Speed** | Slow (disk I/O between every step) | 10-100x faster (in-memory) |
| **Programming** | Verbose Java code | Clean Python/Scala API |
| **Iterative jobs** | Terrible (read-write-read-write...) | Great (keep data in memory) |
| **Use today?** | Rarely directly | Yes — PySpark is the standard |

> **MapReduce is like writing a letter and mailing it between each step.** Spark is like having everyone in the same room talking face-to-face. The ideas are the same — map, shuffle, reduce — but the execution is dramatically faster.

### The Disk vs Memory Problem

```
MapReduce (3-step pipeline):
  Step 1: Read from HDFS → Map → Write to HDFS    (disk I/O)
  Step 2: Read from HDFS → Map → Write to HDFS    (disk I/O)
  Step 3: Read from HDFS → Reduce → Write to HDFS (disk I/O)
  Total: 6 disk operations

Spark (same 3-step pipeline):
  Step 1: Read from HDFS → Transform (in memory)
  Step 2: Transform (in memory)
  Step 3: Transform → Write to HDFS
  Total: 2 disk operations (read once, write once)
```

This is why Spark replaced MapReduce for almost everything. But the **concept** — split, process locally, combine — is the same.

In [ ]:
# ============================================================
# MAPREDUCE — Simulating the Pattern in Python
# ============================================================
from collections import defaultdict

# --- Our "distributed" call center data
# WHY: In real Hadoop, each list below lives on a different physical machine (imagine each list is on a different machine) ---
machine_1_data = [
    {"call_id": "C-001", "campaign": "Solar", "revenue": 149.99},
    {"call_id": "C-002", "campaign": "Auto", "revenue": 0.00},
    {"call_id": "C-003", "campaign": "Solar", "revenue": 89.50},
]
machine_2_data = [
    {"call_id": "C-004", "campaign": "Medicare", "revenue": 210.00},
    {"call_id": "C-005", "campaign": "Solar", "revenue": 125.00},
    {"call_id": "C-006", "campaign": "Auto", "revenue": 75.00},
]
machine_3_data = [
    {"call_id": "C-007", "campaign": "Medicare", "revenue": 185.00},
    {"call_id": "C-008", "campaign": "Solar", "revenue": 0.00},
    {"call_id": "C-009", "campaign": "Auto", "revenue": 110.00},
]

# === STEP 1: MAP — each machine emits (key, value) pairs ===
def map_function(records):
    """Each machine runs this on its local data."""
    return [(r["campaign"], r["revenue"]) for r in records]

print("=== MAP PHASE ===")
mapped_1 = map_function(machine_1_data)
mapped_2 = map_function(machine_2_data)
mapped_3 = map_function(machine_3_data)
print(f"Machine 1 emits: {mapped_1}")
print(f"Machine 2 emits: {mapped_2}")
print(f"Machine 3 emits: {mapped_3}")

# === STEP 2: SHUFFLE — group by key ===
print("\n=== SHUFFLE PHASE ===")
# SHUFFLE: all (key, value) pairs are grouped by key across the network
# WHY: This is the most expensive step — data moves between machines
all_pairs = mapped_1 + mapped_2 + mapped_3
shuffled = defaultdict(list)  # defaultdict auto-creates empty lists for new keys
for key, value in all_pairs:
    shuffled[key].append(value)
for key, values in sorted(shuffled.items()):
    print(f"  {key}: {values}")

# === STEP 3: REDUCE — aggregate each group ===
print("\n=== REDUCE PHASE ===")
for key, values in sorted(shuffled.items()):
    total = sum(values)
    count = len(values)
    print(f"  {key}: count={count}, total_revenue=${total:.2f}")

print("\nThis is EXACTLY what Spark does under the hood when you write:")
print("  df.groupBy('campaign').agg(F.sum('revenue'))")

---

## 4. YARN — The Resource Manager

**One-line answer:** YARN decides which machine runs which task. It's the traffic controller of the Hadoop cluster.

### The Restaurant Manager Analogy

You walk into a busy restaurant. The **host** (YARN ResourceManager) doesn't cook food or serve tables. The host:

1. Knows which tables are open (which machines have free CPU/memory)
2. Assigns your party to a table (assigns your job to machines)
3. Tells the waiter (NodeManager) to take care of you
4. If a waiter calls in sick, reassigns the table to another waiter

That's YARN. It doesn't process data — it **manages who processes data**.

### YARN Architecture

```
+----------------------------------------------+
|           ResourceManager (1 per cluster)     |
|  "I know all available resources.             |
|   I decide where jobs run."                   |
+-----+------------------+---------------------+
      |                  |
      v                  v
+------------+    +------------+    +------------+
| NodeManager|    | NodeManager|    | NodeManager|
| (Machine 1)|    | (Machine 2)|    | (Machine 3)|
|            |    |            |    |            |
| Container  |    | Container  |    | Container  |
| Container  |    | Container  |    | Container  |
| Container  |    |            |    | Container  |
+------------+    +------------+    +------------+
```

### Key Components

| Component | Role | Analogy |
|-----------|------|---------|
| **ResourceManager** | Cluster-wide resource scheduler | Restaurant host — knows all open tables |
| **NodeManager** | Per-machine agent, reports resources | Waiter — manages one section of tables |
| **Container** | Allocated resources (CPU + memory) for a task | A table — reserved for a specific party |
| **ApplicationMaster** | Per-job coordinator | Your waiter — manages your specific order |

### What Happens When You Submit a Job

```
1. You submit: "Run my Spark job"
2. ResourceManager: "OK, I'll allocate an ApplicationMaster on Machine 2"
3. ApplicationMaster starts on Machine 2
4. ApplicationMaster: "I need 10 containers with 4GB RAM each"
5. ResourceManager: "Machine 1 has room for 4, Machine 3 has room for 6"
6. Containers launch on those machines
7. Your Spark tasks run inside those containers
8. When done, containers are released back to the pool
```

### YARN vs Modern Equivalents

| YARN Feature | AWS EMR | GCP Dataproc |
|-------------|---------|--------------|
| ResourceManager | EMR manages this for you | Dataproc manages this for you |
| NodeManager | EC2 instances (auto-provisioned) | Compute Engine VMs |
| Containers | YARN still runs inside EMR! | YARN still runs inside Dataproc! |
| Scaling | Auto-scaling policies | Autoscaler (add/remove workers) |

> **Key insight:** YARN is still running inside EMR and Dataproc. The cloud didn't replace YARN — it just **manages** it for you. You no longer configure `yarn-site.xml` by hand, but the same ResourceManager/NodeManager architecture is running under the hood.

In [ ]:
# ============================================================
# YARN — Resource Allocation Simulation
# ============================================================

# YARN = Yet Another Resource Negotiator — Hadoop's cluster resource manager
# WHY: Without YARN, jobs would fight over CPU/memory. YARN assigns resources fairly.
class YARNSimulator:
    """Simplified YARN resource allocation simulation."""

    def __init__(self, nodes):
                # Initialize cluster with named nodes and their memory capacity
        self.nodes = {name: {"total_gb": gb, "used_gb": 0, "containers": []}
                      for name, gb in nodes.items()}

        # Each job requests containers (CPU+RAM bundles) from the ResourceManager
    def submit_job(self, job_name, containers_needed, gb_per_container):
        print(f"\n[ResourceManager] Job submitted: '{job_name}'")
        print(f"  Requested: {containers_needed} containers x {gb_per_container} GB each")
        print(f"  Total memory needed: {containers_needed * gb_per_container} GB")

        allocated = 0
        for node_name, node in self.nodes.items():
            available = node["total_gb"] - node["used_gb"]
            while available >= gb_per_container and allocated < containers_needed:
                container_id = f"{job_name}-{allocated+1}"
                node["containers"].append(container_id)
                node["used_gb"] += gb_per_container
                available -= gb_per_container
                allocated += 1

        print(f"  Allocated: {allocated}/{containers_needed} containers")
        if allocated < containers_needed:
            print(f"  WARNING: {containers_needed - allocated} containers queued (insufficient resources)")
        self.show_cluster()

        # WHY: When a job finishes, its containers return to the pool for other jobs
    def release_job(self, job_name):
        print(f"\n[ResourceManager] Releasing containers for '{job_name}'")
        for node_name, node in self.nodes.items():
            to_remove = [c for c in node["containers"] if c.startswith(job_name)]
            for c in to_remove:
                node["containers"].remove(c)
                node["used_gb"] -= 4  # simplified
        self.show_cluster()

    def show_cluster(self):
        print("\n  Cluster State:")
        print(f"  {'Node':<15} {'Used':>6} {'Total':>6} {'Containers'}")
        print("  " + "-" * 55)
        for name, node in self.nodes.items():
            containers = ", ".join(node["containers"]) if node["containers"] else "(idle)"
            print(f"  {name:<15} {node['used_gb']:>4} GB {node['total_gb']:>4} GB  {containers}")


# --- Simulate a 3-node cluster ---
# --- Simulate a 3-node cluster (48 GB total = 16 GB per node) ---
# BEGINNER NOTE: In production, nodes have 64-256 GB RAM
yarn = YARNSimulator({
    "DataNode-1": 16,
    "DataNode-2": 16,
    "DataNode-3": 16,
})
print("=== YARN Resource Allocation Demo ===")
yarn.show_cluster()

# Submit a Spark job
yarn.submit_job("spark_etl", containers_needed=6, gb_per_container=4)

# Submit another job (cluster is getting full)
yarn.submit_job("hive_query", containers_needed=4, gb_per_container=4)

# First job finishes
yarn.release_job("spark_etl")

---

## 5. Hive — SQL on Hadoop

**One-line answer:** Hive lets you write SQL queries that run on Hadoop/Spark. It's the SQL interface to big data.

### The Translator Analogy

You're in a foreign country and don't speak the language. A translator sits between you and the locals:

- **You** speak SQL (the language you know)
- **Hive** translates your SQL into MapReduce/Spark jobs (the language the cluster speaks)
- **The cluster** processes the data and returns results

You never have to write Java MapReduce code. You just write SQL.

### What Hive Actually Does

```
You write:
  SELECT campaign, SUM(revenue)
  FROM calls
  WHERE call_date = '2024-01-15'
  GROUP BY campaign;

Hive translates to:
  1. Scan the 'calls' table (stored as Parquet files on HDFS/S3)
  2. Filter: only rows where call_date = '2024-01-15'     ← partition pruning!
  3. Map: emit (campaign, revenue) pairs
  4. Shuffle: group by campaign
  5. Reduce: sum revenue per campaign

You get back a table — just like any SQL database.
```

### Hive Architecture

```
+-----------+     +------------------+     +-------------+
| HiveQL    | --> | Hive Metastore   | --> | Spark/MR    |
| (your SQL)|     | (schema catalog) |     | (execution) |
+-----------+     +------------------+     +------+------+
                  "calls table is at              |
                   s3://bucket/calls/,            v
                   partitioned by date,     +----------+
                   stored as Parquet"       | HDFS / S3|
                                            +----------+
```

### Key Hive Concepts

| Concept | What It Is | Why It Matters |
|---------|-----------|----------------|
| **Metastore** | Database of table schemas (names, types, locations, partitions) | Tables are just metadata pointing to files — Hive doesn't store data itself |
| **Internal table** | Hive manages both metadata AND data files | Drop table = data deleted |
| **External table** | Hive manages metadata only; data stays in S3/HDFS | Drop table = metadata deleted, data survives. **Always use external in production.** |
| **Partitioning** | Organize data into folders by column value | `WHERE date = '2024-01-15'` only reads that folder — called **partition pruning** (skip irrelevant data) |
| **SerDe** | Serializer/Deserializer ("sir-dee") — tells Hive how to read/write the file format | Hive can read CSV, JSON, Parquet, ORC, Avro via different SerDes |

### HiveQL vs Standard SQL

HiveQL is very close to standard SQL, with a few differences:

| Feature | Standard SQL | HiveQL |
|---------|-------------|--------|
| Syntax | Nearly identical | Nearly identical |
| INSERT | `INSERT INTO table VALUES (...)` | `INSERT INTO table SELECT ...` (no VALUES for most formats) |
| UPDATE/DELETE | Supported | Limited (ACID tables only) |
| Subqueries | Everywhere | Most places (some restrictions) |
| Window functions | Supported | Fully supported |
| CTEs (WITH) | Supported | Fully supported |

> **For 95% of your SQL, HiveQL works identically.** The main difference is that Hive is optimized for batch reads over huge datasets, not single-row transactions.

In [ ]:
# ============================================================
# HIVE — HiveQL Examples (syntax reference)
# ============================================================

# These are HiveQL queries you would run on EMR/Dataproc.
# We show them as strings here for reference — you'll run them
# on actual clusters in the hands-on section.

# Hive = SQL interface to Hadoop — translates SQL to MapReduce/Spark jobs
# HiveQL = Hive Query Language (nearly identical to standard SQL)
hive_queries = {
        # --- External tables: the production standard ---
    "Create External Table": """
-- External table: metadata in Hive, data stays in S3
CREATE EXTERNAL TABLE IF NOT EXISTS calls (
    call_id       STRING,
    caller_ani    STRING,
    campaign      STRING,
    duration      INT,
    disposition   STRING,
    converted     BOOLEAN,
    revenue       DOUBLE
)
PARTITIONED BY (call_date STRING)
STORED AS PARQUET
LOCATION 's3://cc-data-lake/bronze/calls/';
-- WHY: EXTERNAL means Hive only tracks metadata. Drop table = data survives in S3.
""",

        # --- Partitions tell Hive where each date's data lives ---
    "Add Partitions": """
-- Tell Hive about existing data in S3
ALTER TABLE calls ADD IF NOT EXISTS
    PARTITION (call_date='2024-01-15')
    LOCATION 's3://cc-data-lake/bronze/calls/call_date=2024-01-15/';
-- Or auto-discover all partitions (scans S3 folders and registers each as a partition):
MSCK REPAIR TABLE calls;  -- MSCK = MetaStore Consistency checK  -- "MSCK" = "metastore check" — syncs partitions with actual folders
""",

        # --- Standard GROUP BY — same SQL you wrote in M03 ---
    "Campaign Summary": """
-- Exactly the SQL you already know from M03
SELECT
    campaign,
    COUNT(*) AS total_calls,
    SUM(CASE WHEN converted THEN 1 ELSE 0 END) AS conversions,
    ROUND(SUM(revenue), 2) AS total_revenue,
    ROUND(AVG(duration), 0) AS avg_duration_sec
FROM calls
WHERE call_date BETWEEN '2024-01-01' AND '2024-01-31'
GROUP BY campaign
ORDER BY total_revenue DESC;
""",

        # --- Window functions work identically in HiveQL and standard SQL ---
    "Window Function": """
-- Rank calls by revenue within each campaign
SELECT
    call_id,
    campaign,
    revenue,
    ROW_NUMBER() OVER (PARTITION BY campaign ORDER BY revenue DESC) AS rank
FROM calls
WHERE converted = true;
""",

        # --- CTAS = Create Table As Select — builds Gold layer marts ---
    "CTAS (Create Table As Select)": """
-- Create a gold mart from raw data
CREATE TABLE gold_campaign_summary
STORED AS PARQUET
LOCATION 's3://cc-data-lake/gold/campaign_summary/'
AS
SELECT
    campaign,
    call_date,
    COUNT(*) AS calls,
    SUM(revenue) AS revenue
FROM calls
GROUP BY campaign, call_date;
""",
}

# Display each query
# BEGINNER NOTE: These are reference examples — you'll run real HiveQL on EMR
# Print each query with a header for easy reference
for name, query in hive_queries.items():
    print(f"--- {name} ---")
    print(query.strip())
    print()

In [ ]:
# ============================================================
# HIVE vs SPARK SQL — Side-by-Side
# ============================================================

# The point: HiveQL and Spark SQL are nearly identical.
# If you know SQL (you do from M03), you already know HiveQL.

# WHY: Proving that HiveQL and Spark SQL are nearly identical
# Your M03 SQL skills transfer directly — no new syntax to learn
comparisons = [
    ("Filter + Aggregate",
     # HiveQL (runs on EMR Hive)
     """SELECT campaign, COUNT(*) as calls, SUM(revenue) as rev
FROM calls
WHERE call_date = '2024-01-15'
GROUP BY campaign;""",
     # Spark SQL (runs in PySpark)
     """spark.sql(\"\"\"
    SELECT campaign, COUNT(*) as calls, SUM(revenue) as rev
    FROM calls
    WHERE call_date = '2024-01-15'
    GROUP BY campaign
\"\"\")""",
    ),
    ("Window Function",
     # HiveQL
     """SELECT call_id, campaign, revenue,
       RANK() OVER (PARTITION BY campaign ORDER BY revenue DESC) as rnk
FROM calls;""",
     # Spark SQL
     """spark.sql(\"\"\"
    SELECT call_id, campaign, revenue,
           RANK() OVER (PARTITION BY campaign ORDER BY revenue DESC) as rnk
    FROM calls
\"\"\")""",
    ),
]

for name, hive_q, spark_q in comparisons:
    print(f"=== {name} ===")
    print(f"\n  HiveQL (on EMR):")
    for line in hive_q.strip().split("\n"):
        print(f"    {line}")
    print(f"\n  Spark SQL (in PySpark):")
    for line in spark_q.strip().split("\n"):
        print(f"    {line}")
    print(f"\n  Difference: Almost none. The SQL is the same.")
    print()

print("Key takeaway: Learn SQL well (M03) and you can use it everywhere —")
print("Hive, Spark SQL, BigQuery, Athena. The engine changes, the SQL stays.")

---

## 6. The Legacy-to-Modern Mapping

**One-line answer:** Every Hadoop component has a modern managed equivalent. The architecture is the same — the operations burden disappeared.

### The Full Mapping

| Legacy (On-Prem) | Modern Managed (Cloud) | What Changed |
|-------------------|----------------------|--------------|
| **HDFS** | S3 / GCS / ADLS | Storage decoupled from compute. Pay per GB. Infinite scale. |
| **MapReduce** | Spark on EMR / Dataproc | In-memory processing. 10-100x faster. Python API (PySpark). |
| **Hive on Hadoop** | Hive on EMR / Athena / BigQuery | Same SQL, managed infrastructure. Serverless options. |
| **YARN** | EMR cluster management / Dataproc autoscaler | Still YARN under the hood, but auto-configured. |
| **Hadoop Admin** | Click "Create Cluster" | No more tuning hdfs-site.xml, yarn-site.xml, etc. |
| **Rack awareness** | Availability zones | Cloud handles data placement across zones. |
| **NameNode HA** | Managed by EMR/Dataproc | Automatic failover, no manual standby setup. |

### The Three Big Shifts

```
Shift 1: STORAGE DECOUPLED FROM COMPUTE
  Before: Data on HDFS (same machines that process)
  After:  Data on S3/GCS (separate from processing machines)
  Why:    Scale storage and compute independently. Shut down cluster, data persists.

Shift 2: EPHEMERAL CLUSTERS (short-lived, disposable)
  Before: Cluster runs 24/7 (even when idle — you're paying)
  After:  Spin up cluster → run job → tear down (pay only for compute time)
  Why:    10x cost savings. No idle machines.

Shift 3: SERVERLESS OPTIONS
  Before: You manage the cluster (even if cloud-hosted)
  After:  Athena/BigQuery — no cluster at all. Just run SQL.
  Why:    Zero operations. Pay per query.
```

### When to Use What (Decision Tree)

```
"I need to process data"
  |
  +-- How much data?
       |
       +-- < 1 GB          → pandas on your laptop
       |
       +-- 1 GB - 100 GB   → PySpark local or small EMR cluster
       |
       +-- 100 GB - 10 TB  → EMR / Dataproc cluster
       |
       +-- > 10 TB         → EMR / Dataproc + S3/GCS
       |
       +-- Just SQL?        → Athena (serverless, pay-per-query)
       |                      BigQuery (serverless, pay-per-query)
       |
       +-- Real-time?       → Kinesis / Pub/Sub + Spark Streaming
```

> **Bottom line:** You're not learning Hadoop because you'll run HDFS on-prem. You're learning it because **EMR is Hadoop, managed. Dataproc is Hadoop, managed. Athena is Hive, serverless.** Understanding the architecture lets you use the tools intelligently.

In [ ]:
# ============================================================
# LEGACY-TO-MODERN — Architecture Comparison
# ============================================================

# WHY: Understanding the shift from on-prem to cloud is a key interview topic
# Visual comparison of on-prem Hadoop vs cloud-managed

# WHY: This visual comparison is the key slide in any "why cloud?" discussion
# BEGINNER NOTE: You don't need to memorize this — understand the SHIFT
# Display the results so you can verify the output matches expectations
print("""
=============================================================
  ON-PREM HADOOP CLUSTER (2012 era)
=============================================================

  +------ Your Data Center ------+
  |                              |
  |  NameNode (you maintain)     |
  |  ResourceManager (you tune)  |
  |                              |
  |  +--------+  +--------+     |
  |  |DataNode|  |DataNode|     |  <- Your servers
  |  |+ HDFS  |  |+ HDFS  |     |  <- Data AND compute coupled
  |  |+ YARN  |  |+ YARN  |     |  <- You configure everything
  |  +--------+  +--------+     |
  |                              |
  |  Hive Metastore (you run)    |
  |  ZooKeeper (you manage)      |
  +------------------------------+

  You manage: hardware, OS, Hadoop config, security,
  monitoring, backups, scaling, failures...

=============================================================
  AWS EMR + S3 (2024 era)
=============================================================

  +------ S3 (managed storage) ------+
  |  s3://your-data-lake/            |
  |    bronze/calls/                 |  <- Infinite, durable
  |    silver/calls_clean/           |  <- Persists even if
  |    gold/campaign_summary/        |     cluster is deleted
  +----------------------------------+
           |
           v (reads/writes)
  +------ EMR Cluster (ephemeral) ---+
  |                                  |
  |  Master: NameNode + YARN         |  <- Auto-configured
  |  Worker 1: Spark executor        |  <- Auto-scaled
  |  Worker 2: Spark executor        |
  |  Worker 3: Spark executor        |
  |                                  |
  |  Hive Metastore: AWS Glue Catalog|  <- Serverless
  +----------------------------------+
       (spin up → run job → tear down)

  You manage: your code. That's it.
""")

# --- Cost comparison: the #1 reason companies migrated to cloud ---
print("--- Cost Comparison (approximate) ---")
print(f"{'Scenario':<35} {'On-Prem Hadoop':>15} {'AWS EMR':>15}")
# Separator line
print("-" * 67)
# Real-world cost comparison — these numbers explain why cloud won
scenarios = [
    ("10-node cluster (annual)", "$500K-$1M", "$50K-$150K"),
    ("Admin staff (2 Hadoop admins)", "$300K/year", "$0 (managed)"),
    ("Idle cluster cost", "Same as running", "$0 (shut down)"),
    ("Scale up (add 10 nodes)", "Weeks (order HW)", "Minutes (API call)"),
    ("Data durability", "3x replication", "11 nines (S3)"),
]
# WHY: These numbers explain why on-prem Hadoop is nearly extinct in 2026
for scenario, onprem, cloud in scenarios:
    print(f"{scenario:<35} {onprem:>15} {cloud:>15}")

---

## 7. AWS EMR — Managed Hadoop in the Cloud

**One-line answer:** EMR is Hadoop-as-a-service. You get Spark, Hive, and YARN without managing any infrastructure.

### What EMR Gives You

| Component | How EMR Handles It |
|-----------|-------------------|
| **HDFS** | Optional (uses EMRFS → S3 instead) |
| **Spark** | Pre-installed, configured, version-managed |
| **Hive** | Pre-installed + integrated with AWS Glue Catalog |
| **YARN** | Auto-configured based on instance types |
| **Scaling** | Managed auto-scaling (add/remove instances) |
| **Security** | IAM roles, encryption, VPC, Kerberos |
| **Monitoring** | CloudWatch metrics + Spark UI |

### EMR Cluster Types

| Node Type | Role | Scaling |
|-----------|------|---------|
| **Master** | Runs NameNode, ResourceManager, Hive Metastore | 1 (or 3 for HA) — fixed |
| **Core** | Runs DataNode + NodeManager (stores HDFS data + processes) | Can scale up/down |
| **Task** | Runs NodeManager only (processes, no storage) | Elastic — scale freely |

> **Restaurant analogy:** Master = the manager (always there, runs the operation). Core = full-time cooks (always there, they store recipes AND cook). Task = temp workers for a busy night (just cook, don't store anything, easy to add/remove).

> **Pattern:** Use a small number of Core nodes for stability + many Task nodes for burst processing. Task nodes can use Spot instances (80% cheaper).

### EMR Lifecycle: The Ephemeral Pattern

```
Step 1: Create cluster          (2-5 minutes)
Step 2: Submit Spark/Hive jobs  (minutes to hours)
Step 3: Results written to S3   (persists forever)
Step 4: Terminate cluster       (stop paying)

Your data lives in S3 — the cluster is disposable.
Spin up a new one tomorrow with different settings.
```

### Key EMR Concepts

| Concept | What It Means | Analogy |
|---------|--------------|---------|
| **Step** | A job submitted to the cluster (Spark script, Hive query) | An order at the restaurant |
| **Bootstrap action** | Script that runs on every node at startup | Setting up the kitchen before opening |
| **EMRFS** | S3 connector that makes S3 look like HDFS to Spark/Hive | An adapter plug — same tools, different outlet |
| **AWS Glue Catalog** | Managed Hive Metastore — stores table schemas | The catalog of all your tables |
| **Spot instances** | Cheap EC2 instances AWS can reclaim | Standby seats on a flight — cheap but not guaranteed |

In [ ]:
# ============================================================
# AWS EMR — CLI Commands Reference
# ============================================================

# These are the AWS CLI commands you'll use to manage EMR clusters.
# In production, these run from your terminal, CI/CD, or Airflow DAG.

# EMR = Elastic MapReduce — AWS's managed Hadoop/Spark service
# WHY: These CLI commands are what you'd run in production or from an Airflow DAG
emr_commands = {
        # --- Step 1: Create cluster (most flags are self-explanatory) ---
    "Create a cluster": """
aws emr create-cluster \\
    --name "DE-Call-Center-ETL" \\
    --release-label emr-7.0.0 \\
    --applications Name=Spark Name=Hive \\
    --instance-groups \\
        InstanceGroupType=MASTER,InstanceCount=1,InstanceType=m5.xlarge \\
        InstanceGroupType=CORE,InstanceCount=2,InstanceType=m5.xlarge \\
        InstanceGroupType=TASK,InstanceCount=3,InstanceType=m5.xlarge \\
    --use-default-roles \\
    --log-uri s3://cc-logs/emr/ \\
    --auto-terminate
""",

        # --- Step 2: Submit work to the cluster ---
    "Submit a Spark job (Step)": """
aws emr add-steps \\
    --cluster-id j-XXXXXXXXXXXXX \\
    --steps Type=Spark,Name="Daily ETL",\\
ActionOnFailure=CONTINUE,\\
Args=[--deploy-mode,cluster,\\
s3://cc-scripts/etl/daily_calls_etl.py]
""",

        # --- Hive queries run as Steps too (same pattern as Spark) ---
    "Submit a Hive query": """
aws emr add-steps \\
    --cluster-id j-XXXXXXXXXXXXX \\
    --steps Type=Hive,Name="Campaign Summary",\\
ActionOnFailure=CONTINUE,\\
Args=[-f,s3://cc-scripts/hive/campaign_summary.hql]
""",

        # --- Monitoring commands ---
    "Check cluster status": """
aws emr describe-cluster --cluster-id j-XXXXXXXXXXXXX
""",

    "List running clusters": """
aws emr list-clusters --active
""",

        # --- Step 3: Tear down when done (stop paying!) ---
    "Terminate cluster": """
aws emr terminate-clusters --cluster-ids j-XXXXXXXXXXXXX
""",

    "SSH into master node": """
aws emr ssh --cluster-id j-XXXXXXXXXXXXX --key-pair-file ~/my-key.pem
""",
}

# Display each command with its name as header
for name, cmd in emr_commands.items():
    print(f"--- {name} ---")
    print(cmd.strip())
    print()

print("Tip: Use --auto-terminate to shut down after all Steps complete.")
print("Tip: Use Spot instances for Task nodes to save 60-80% on compute.")

In [ ]:
# ============================================================
# AWS EMR — Cost Estimation
# ============================================================

# Real-world EMR pricing (approximate, us-east-1, 2024)
# On-Demand prices — Spot instances are 60-80% cheaper

# WHY: Understanding pricing is critical — cloud bills can surprise you
instance_prices = {
    "m5.xlarge":  {"vcpu": 4,  "ram_gb": 16, "hourly": 0.192},
    "m5.2xlarge": {"vcpu": 8,  "ram_gb": 32, "hourly": 0.384},
    "m5.4xlarge": {"vcpu": 16, "ram_gb": 64, "hourly": 0.768},
    "r5.xlarge":  {"vcpu": 4,  "ram_gb": 32, "hourly": 0.252},  # memory-optimized
}

# EMR surcharge: ~25% on top of EC2 price
EMR_SURCHARGE = 0.25  # AWS charges 25% on top of EC2 price for EMR management

print("--- EMR Instance Pricing (On-Demand, us-east-1) ---")
print(f"{'Instance':<15} {'vCPU':>5} {'RAM':>6} {'EC2/hr':>8} {'EMR/hr':>8} {'EMR/day':>9}")
print("-" * 55)
for name, specs in instance_prices.items():
    emr_hr = specs["hourly"] * (1 + EMR_SURCHARGE)
    emr_day = emr_hr * 24
    print(f"{name:<15} {specs['vcpu']:>5} {specs['ram_gb']:>4} GB ${specs['hourly']:>6.3f} ${emr_hr:>6.3f} ${emr_day:>7.2f}")

# Example cluster cost
print("\n--- Example Cluster: 1 Master + 2 Core + 3 Task (m5.xlarge) ---")
nodes = {"Master": 1, "Core": 2, "Task (Spot)": 3}
spot_discount = 0.3  # Spot instances = ~70% cheaper (AWS can reclaim them anytime)  # Spot price is ~30% of On-Demand
base = instance_prices["m5.xlarge"]["hourly"] * (1 + EMR_SURCHARGE)

# Calculate total cluster cost per hour
total_hourly = 0
for role, count in nodes.items():
    if "Spot" in role:
        rate = base * spot_discount
    else:
        rate = base
    subtotal = rate * count
    total_hourly += subtotal
    print(f"  {role}: {count} x ${rate:.3f}/hr = ${subtotal:.3f}/hr")

print(f"\n  Total: ${total_hourly:.3f}/hr = ${total_hourly * 8:.2f}/day (8hr job)")
print(f"  Monthly (8hr/day, 22 days): ${total_hourly * 8 * 22:.2f}")
print(f"\n  vs. Always-on 24/7: ${total_hourly * 24 * 30:.2f}/month")
print(f"  Ephemeral pattern saves: {(1 - (8*22)/(24*30))*100:.0f}%")

---

## 8. GCP Dataproc — Google's Equivalent

**One-line answer:** Dataproc is Google's EMR. Same concepts, different CLI, tighter BigQuery integration.

### EMR vs Dataproc — Quick Comparison

| Feature | AWS EMR | GCP Dataproc |
|---------|---------|--------------|
| **Storage** | S3 (EMRFS) | GCS (Cloud Storage connector) |
| **Metastore** | Glue Catalog | Dataproc Metastore (or Hive on cluster) |
| **Spark** | Pre-installed | Pre-installed |
| **Hive** | Pre-installed | Pre-installed |
| **Serverless SQL** | Athena | BigQuery |
| **Cluster startup** | 5-10 minutes | 90 seconds |
| **Auto-scaling** | Managed scaling | Autoscaler (aggressive) |
| **Spot/Preemptible** | Spot instances | Spot VMs |
| **Notebooks** | EMR Studio / Jupyter | Jupyter on Dataproc |
| **Pricing model** | Per-second + EMR surcharge | Per-second, lower base |

### Key Dataproc Commands

```bash
# Create a cluster
gcloud dataproc clusters create de-cluster \
    --region us-central1 \
    --master-machine-type n1-standard-4 \
    --worker-machine-type n1-standard-4 \
    --num-workers 3 \
    --image-version 2.1

# Submit a PySpark job
gcloud dataproc jobs submit pyspark \
    gs://cc-scripts/etl/daily_calls_etl.py \
    --cluster de-cluster \
    --region us-central1

# Submit a Hive query
gcloud dataproc jobs submit hive \
    --cluster de-cluster \
    --region us-central1 \
    -f gs://cc-scripts/hive/campaign_summary.hql

# Delete cluster
gcloud dataproc clusters delete de-cluster --region us-central1
```

> **For this program:** We use GCP (BigQuery, GCS, Dataproc) as our primary cloud. The Hadoop concepts are identical — only the CLI changes.

### The Serverless Option: BigQuery

BigQuery is what you get when you take Hive's idea (SQL on big data) and make it truly serverless:

| | Hive on EMR/Dataproc | BigQuery |
|---|---------------------|----------|
| **Cluster** | You create and manage one | None — fully serverless |
| **Startup** | 2-10 minutes | 0 seconds |
| **Scaling** | You configure auto-scaling | Automatic |
| **Cost** | Pay for cluster time | Pay per TB scanned |
| **Best for** | Complex ETL, Spark jobs | Ad-hoc queries, analytics |

> BigQuery is covered in M08. Think of it as: Hive evolved into serverless SQL.

---

## 9. Hands-On: The EMR Workflow

**One-line answer:** Spin up a cluster, run a Hive query on S3 data, check results, tear it down.

### The Full Workflow

```
Step 1: Upload data to S3
  aws s3 cp calls.parquet s3://cc-data-lake/bronze/calls/

Step 2: Create EMR cluster
  aws emr create-cluster ... --applications Name=Spark Name=Hive

Step 3: Connect to cluster
  aws emr ssh --cluster-id j-XXXX --key-pair-file key.pem

Step 4: Run Hive queries
  hive> CREATE EXTERNAL TABLE calls ... LOCATION 's3://cc-data-lake/bronze/calls/';
  hive> SELECT campaign, SUM(revenue) FROM calls GROUP BY campaign;

Step 5: Run PySpark job
  spark-submit s3://cc-scripts/etl/daily_calls_etl.py

Step 6: Verify output in S3
  aws s3 ls s3://cc-data-lake/gold/campaign_summary/

Step 7: Terminate cluster
  aws emr terminate-clusters --cluster-ids j-XXXX
```

### What You'll Do in Lab

We'll walk through a simplified version using PySpark locally (simulating what EMR does), then discuss the full cloud workflow:

1. **Create sample Parquet data** (simulating S3 upload)
2. **Create SparkSession** (simulating EMR cluster)
3. **Register as Hive-style table** (simulating Glue Catalog)
4. **Run HiveQL queries via spark.sql()** (same SQL, local execution)
5. **Write results as partitioned Parquet** (simulating S3 output)
6. **Stop SparkSession** (simulating cluster termination)

In [ ]:
# ============================================================
# HANDS-ON — Step 1: Create Sample Data (simulating S3 upload)
# ============================================================
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import os

# --- Create SparkSession (simulating EMR cluster) ---
# WHY: SparkSession is your entry point to all Spark functionality
# local[*] = use all CPU cores on this machine (simulates a cluster)
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("M05-EMR-Simulation") \
    .config("spark.sql.warehouse.dir", "/tmp/spark-warehouse") \
    .enableHiveSupport()  # Enables HiveQL syntax in spark.sql() queries \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print(f"SparkSession created (simulating EMR cluster)")
print(f"Spark version: {spark.version}")

# --- Create call center data ---
# Sample call center data — same schema you'll use on real EMR clusters
calls_data = [
    ("C-001", "2024-01-15", "Solar", 342, "converted", True, 149.99),
    ("C-002", "2024-01-15", "Auto", 45, "no_sale", False, 0.00),
    ("C-003", "2024-01-15", "Solar", 210, "converted", True, 89.50),
    ("C-004", "2024-01-16", "Auto", 18, "abandoned", False, 0.00),
    ("C-005", "2024-01-16", "Solar", 520, "no_sale", False, 0.00),
    ("C-006", "2024-01-16", "Medicare", 180, "converted", True, 210.00),
    ("C-007", "2024-01-17", "Medicare", 90, "no_sale", False, 0.00),
    ("C-008", "2024-01-17", "Solar", 275, "converted", True, 125.00),
    ("C-009", "2024-01-17", "Auto", 150, "converted", True, 75.00),
    ("C-010", "2024-01-17", "Solar", 60, "no_sale", False, 0.00),
]

# StructType = PySpark's way of defining column names and types
# WHY: Explicit schemas prevent type inference errors on large datasets
schema = StructType([
    StructField("call_id", StringType()),
    StructField("call_date", StringType()),
    StructField("campaign", StringType()),
    StructField("duration", IntegerType()),
    StructField("disposition", StringType()),
    StructField("converted", BooleanType()),
    StructField("revenue", DoubleType()),
])

calls_df = spark.createDataFrame(calls_data, schema)

# --- Write as partitioned Parquet (simulating S3 storage) ---
output_path = "/tmp/m05_data_lake/bronze/calls"
calls_df.write.mode("overwrite").partitionBy("call_date").parquet(output_path)

print(f"\nWrote {calls_df.count()} records to {output_path}")
print("Partitions:")
for d in sorted(os.listdir(output_path)):
    if d.startswith("call_date"):
        print(f"  {d}/")

In [ ]:
# ============================================================
# HANDS-ON — Step 2: Create Hive-style Tables (simulating Glue Catalog)
# ============================================================

# --- Register as a Hive table (this is what Glue Catalog does on EMR) ---
calls_from_parquet = spark.read.parquet(output_path)
calls_from_parquet.createOrReplaceTempView("calls")

# In EMR, you'd write:
#   CREATE EXTERNAL TABLE calls (...) PARTITIONED BY (call_date STRING)
#   STORED AS PARQUET LOCATION 's3://cc-data-lake/bronze/calls/';
#   MSCK REPAIR TABLE calls;  -- scans S3 folders, registers each as a Hive partition

print("Table 'calls' registered.")
print(f"Columns: {calls_from_parquet.columns}")
print(f"Rows: {calls_from_parquet.count()}")
spark.sql("DESCRIBE calls").show(truncate=False)

In [ ]:
# ============================================================
# HANDS-ON — Step 3: Run HiveQL Queries (same SQL as EMR)
# ============================================================

# --- Query 1: Campaign Summary ---
# WHY: This is the #1 report every call center needs — revenue by campaign
print("=== Campaign Summary ===")
spark.sql("""
    SELECT
        campaign,
        COUNT(*) AS total_calls,
        SUM(CASE WHEN converted THEN 1 ELSE 0 END) AS conversions,
        ROUND(SUM(CASE WHEN converted THEN 1.0 ELSE 0.0 END) / COUNT(*) * 100, 1) AS conv_rate_pct,
        ROUND(SUM(revenue), 2) AS total_revenue,
        ROUND(AVG(duration), 0) AS avg_duration_sec
    FROM calls
    GROUP BY campaign
    ORDER BY total_revenue DESC
""").show(truncate=False)

# --- Query 2: Daily Trend ---
# WHY: Daily trends show if campaigns are improving or declining
print("=== Daily Trend ===")
spark.sql("""
    SELECT
        call_date,
        COUNT(*) AS calls,
        SUM(CASE WHEN converted THEN 1 ELSE 0 END) AS conversions,
        ROUND(SUM(revenue), 2) AS revenue
    FROM calls
    GROUP BY call_date
    ORDER BY call_date
""").show(truncate=False)

# --- Query 3: Window Function (same syntax as M03 — ROW_NUMBER + PARTITION BY) — Rank by Revenue per Campaign ---
print("=== Top Calls by Revenue per Campaign ===")
spark.sql("""
    SELECT call_id, campaign, revenue,
           ROW_NUMBER() OVER (PARTITION BY campaign ORDER BY revenue DESC) AS rank
    FROM calls
    WHERE converted = true
""").show(truncate=False)

In [ ]:
# ============================================================
# HANDS-ON — Step 4: Create Gold Mart (simulating CTAS to S3)
# ============================================================

# This is exactly what you'd do on EMR:
#   CREATE TABLE gold_campaign_summary STORED AS PARQUET
#   LOCATION 's3://cc-data-lake/gold/campaign_summary/' AS SELECT ...

gold_df = spark.sql("""
    SELECT
        campaign,
        call_date,
        COUNT(*) AS total_calls,
        SUM(CASE WHEN converted THEN 1 ELSE 0 END) AS conversions,
        ROUND(SUM(revenue), 2) AS total_revenue,
        ROUND(AVG(duration), 0) AS avg_duration
    FROM calls
    GROUP BY campaign, call_date
    ORDER BY campaign, call_date
""")

# Write as gold mart
gold_path = "/tmp/m05_data_lake/gold/campaign_summary"
gold_df.coalesce(1).write.mode("overwrite").parquet(gold_path)

print("=== Gold Mart: Campaign Summary ===")
gold_df.show(truncate=False)
print(f"\nWritten to: {gold_path}")
print("On EMR, this would be: s3://cc-data-lake/gold/campaign_summary/")

In [ ]:
# ============================================================
# HANDS-ON — Step 5: Verify + Cleanup (simulating cluster termination)
# ============================================================

# --- Verify gold mart ---
print("=== Verify Gold Mart (read back from Parquet) ===")
verify = spark.read.parquet(gold_path)
print(f"Rows: {verify.count()}")
print(f"Schema:")
verify.printSchema()
verify.show(truncate=False)

# --- Show the full pipeline ---
print("""
=== Full EMR Workflow Summary ===

What we simulated locally:           What it looks like on EMR:
-----------------------------------------+-------------------------------------------
1. spark = SparkSession(local)            | aws emr create-cluster ...
2. df.write.parquet("/tmp/...")            | aws s3 cp calls.parquet s3://bucket/...
3. createOrReplaceTempView("calls")       | CREATE EXTERNAL TABLE ... LOCATION 's3://...'
4. spark.sql("SELECT ... GROUP BY ...")   | hive -e "SELECT ... GROUP BY ..."
5. gold.write.parquet("/tmp/gold/...")     | Results written to s3://bucket/gold/...
6. # WHY: Always stop SparkSession when done — releases cluster resources
# On EMR: this is like terminating the cluster (stop paying for compute)
spark.stop()                           | aws emr terminate-clusters --cluster-ids ...
""")

# --- Stop SparkSession (simulating cluster termination) ---
# WHY: Always stop SparkSession when done — releases cluster resources
# On EMR: this is like terminating the cluster (stop paying for compute)
spark.stop()
print("SparkSession stopped (cluster terminated).")
print("Data persists in '/tmp/m05_data_lake/' (simulating S3).")
print("\nIn production: data persists in S3 forever. Cluster is gone. You only paid for compute time.")

---## Hadoop & Distributed Systems Interview QuestionsThese come up in every DE interview, even in 2026. The concepts haven't changed — only the management layer has.| # | Question | Answer ||---|----------|--------|| 1 | **NameNode vs DataNode?** | NameNode = the librarian's card catalog (knows where every block is stored). DataNode = the bookshelves (stores actual 128 MB data blocks). NameNode is metadata only — never touches the data itself. || 2 | **What is HDFS block size and why 128 MB?** | Default block size is 128 MB (was 64 MB in early Hadoop). WHY: Too small → NameNode tracks millions of blocks (memory problem). Too large → wasted space for small files. 128 MB balances metadata overhead vs storage efficiency. || 3 | **What is the default replication factor?** | 3. Every block is stored on 3 different DataNodes. If one machine dies, data survives on the other 2. HDFS automatically re-replicates to restore factor of 3. In cloud (S3/GCS), replication is handled for you (11 nines durability). || 4 | **What happens when a DataNode fails?** | NameNode detects the failure (missed heartbeats). It finds which blocks were on that node, locates replicas on other nodes, and triggers re-replication to restore the replication factor. No data is lost if replication factor >= 2. || 5 | **Hive vs Spark SQL?** | Hive was the original SQL-on-Hadoop (translates SQL → MapReduce). Spark SQL is faster (in-memory). Both use nearly identical SQL syntax. Hive is still used for metadata (Metastore/Glue Catalog). Most teams run Spark SQL with Hive Metastore. || 6 | **When is Hadoop still relevant in 2026?** | The CONCEPTS are everywhere: block storage (S3), MapReduce pattern (Spark), resource management (YARN inside EMR/Dataproc), SQL-on-files (Hive → Athena/BigQuery). You're not running HDFS on-prem, but you're using Hadoop's architecture every day in the cloud. || 7 | **What is the difference between vertical and horizontal scaling?** | Vertical = bigger machine (more CPU/RAM). Has a ceiling. Horizontal = more machines, split the data. No ceiling. Hadoop chose horizontal — 1,000 cheap machines instead of 1 expensive supercomputer. || 8 | **What is an ephemeral cluster?** | A cluster you spin up, run your job, and tear down. Data lives in S3/GCS (persists). Compute is temporary (pay only while running). Saves 70-80% vs always-on clusters. This is the modern pattern on EMR/Dataproc. |> **Interview tip:** Always connect your answer to a real example. "In our call center pipeline, we stored raw call records in S3 (Bronze layer) and spun up an EMR cluster to run PySpark ETL. After processing, the cluster terminated — we only paid for 4 hours of compute."


> **Key terms:** **ETL** = Extract, Transform, Load (transform data BEFORE loading into the warehouse — clean it first, then store it). **ELT** = Extract, Load, Transform (load raw data first, then transform inside the warehouse — BigQuery/Snowflake are powerful enough to transform after loading). Modern cloud pipelines mostly use ELT because cloud warehouses handle transformation efficiently. Our medallion pipeline (Bronze → Silver → Gold) is an ELT pattern: we load raw data to Bronze (Extract + Load), then transform in Silver/Gold (Transform).


---

## 10. Key Takeaways + Homework

### Key Takeaways

1. **Hadoop solved three problems:** storage (HDFS), processing (MapReduce), coordination (YARN)
2. **HDFS** splits files into 128 MB blocks, replicates 3x across machines — fault-tolerant by design
3. **MapReduce** = map (process locally in parallel) + shuffle (group by key) + reduce (aggregate). Spark does the same thing, 10-100x faster, in memory
4. **YARN** is the resource manager — decides which machine runs which task. Still runs inside EMR/Dataproc
5. **Hive** = SQL on big data. Your M03 SQL skills transfer directly to HiveQL
6. **External tables** point to data in S3/GCS — drop the table, data survives. Always use external in production
7. **EMR/Dataproc** = Hadoop managed by the cloud. Same architecture, zero operations
8. **Ephemeral clusters** = spin up, run job, tear down. Pay only for compute time
9. **Storage decoupled from compute** = data in S3/GCS persists independently of cluster lifecycle
10. **The SQL stays the same** — whether you run it in Hive, Spark SQL, Athena, or BigQuery

### Homework

| Task | Time | Description |
|------|------|-------------|
| **1. AWS EMR Documentation** | 30 min | Read [EMR Getting Started](https://docs.aws.amazon.com/emr/latest/ManagementGuide/emr-gs.html) — focus on cluster creation and Step submission |
| **2. HiveQL Practice** | 30 min | Write 5 HiveQL queries against the call center schema (use M03 queries as starting points — they're the same SQL!) |
| **3. Architecture Diagram** | 20 min | Draw the flow: S3 (bronze) → EMR (Spark/Hive) → S3 (gold) → BigQuery (analytics). Label each component with its Hadoop equivalent |
| **4. Cost Exercise** | 15 min | Calculate: 3-node EMR cluster (m5.xlarge), runs 4 hours/day, 22 days/month. Compare On-Demand vs Spot pricing |

### Preview: Module 06

In M06, you'll write PySpark code that processes our call center data — the same kind of code that would run on EMR. The Python for Data Engineering notebook covers PySpark fundamentals in depth.